In [1]:
import pandas as pd
import datetime
import re

# -----------------------------
# Helper functions
# -----------------------------
month_map = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4, "MAY": 5,
    "JUN": 6, "JUL": 7, "AUG": 8, "SEP": 9, "OCT": 10,
    "NOV": 11, "DEC": 12
}

def parse_lesson_log(raw_log, start_year):
    """
    Parses a raw log into a structured DataFrame.
    raw_log: list of strings (month/day/weekday/time/status)
    start_year: int, year the student started
    """
    # Expecting chunks of 5 lines per lesson: month, day, weekday, time, status
    data = []
    for i in range(0, len(raw_log), 5):
        try:
            month = raw_log[i].strip()
            day = int(raw_log[i+1].strip())
            weekday = raw_log[i+2].strip()
            time_slot = raw_log[i+3].strip()
            status = raw_log[i+4].strip()
            
            # Extract payment
            payment_match = re.search(r'\$([0-9]+\.[0-9]+)', status)
            payment = float(payment_match.group(1)) if payment_match else 0.0
            
            # Confirmed / Cancelled
            if "Cancel" in status:
                lesson_status = "Cancelled"
            else:
                lesson_status = "Confirmed"
            
            # Date
            month_num = month_map[month.upper()]
            date = datetime.date(start_year, month_num, day)
            
            data.append({
                "date": date,
                "month": month,
                "day": day,
                "weekday": weekday,
                "time_slot": time_slot,
                "status": lesson_status,
                "payment": payment
            })
        except Exception as e:
            print(f"Error parsing lines {i}-{i+4}: {e}")
            continue

    df = pd.DataFrame(data)
    df.sort_values('date', inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

# -----------------------------
# Compute summary metrics
# -----------------------------
def student_summary(df):
    total_lessons = len(df)
    confirmed_lessons = df[df['status']=="Confirmed"].shape[0]
    cancelled_lessons = df[df['status']=="Cancelled"].shape[0]
    total_revenue = df['payment'].sum()
    
    # Average lessons per week
    weeks_active = (df['date'].max() - df['date'].min()).days / 7
    avg_lessons_per_week = confirmed_lessons / weeks_active if weeks_active > 0 else 0
    
    # Cancellation rate
    cancellation_rate = cancelled_lessons / total_lessons * 100 if total_lessons > 0 else 0
    
    # Lessons per month
    lessons_per_month = df.groupby('month').size().to_dict()
    
    # Lessons per weekday
    lessons_per_weekday = df.groupby('weekday').size().to_dict()
    
    summary = {
        "total_lessons": total_lessons,
        "confirmed_lessons": confirmed_lessons,
        "cancelled_lessons": cancelled_lessons,
        "total_revenue": total_revenue,
        "avg_lessons_per_week": round(avg_lessons_per_week,2),
        "cancellation_rate (%)": round(cancellation_rate,2),
        "lessons_per_month": lessons_per_month,
        "lessons_per_weekday": lessons_per_weekday
    }
    
    return summary


In [2]:
raw_log = [
   AUG
30
Friday
12:30 PM – 1:20 PM
Cancelled
JUN
21
Friday
1:00 PM – 1:50 PM
Cancelled
JUN
10
Monday
1:00 PM – 1:50 PM
Cancelled
MAY
1
Wednesday
12:00 PM – 12:50 PM
Cancelled
APR
25
Thursday
11:00 AM – 11:50 AM
Cancelled
MAR
28
Thursday
12:30 PM – 1:20 PM
Cancelled
FEB
4
Sunday
11:00 AM – 11:50 AM
Confirmed + $13.94
DEC
17
Sunday
11:30 AM – 12:20 PM
Cancelled
DEC
10
Sunday
12:00 PM – 12:50 PM
Cancelled
DEC
10
Sunday
11:00 AM – 11:50 AM
Cancelled
SEP
21
Thursday
12:30 PM – 1:20 PM
Cancelled
SEP
18
Monday
12:30 PM – 1:20 PM
Cancelled
JUL
20
Thursday
12:30 PM – 1:20 PM
Cancelled
JUL
5
Wednesday
11:00 AM – 11:50 AM
Confirmed + $13.94
JUL
3
Monday
11:30 AM – 12:20 PM
Confirmed + $13.94
JUN
30
Friday
12:00 PM – 12:50 PM
Confirmed + $13.94
JUN
25
Sunday
11:30 AM – 12:20 PM
Confirmed + $13.94
JUN
22
Thursday
12:30 PM – 1:20 PM
Confirmed + $13.94
JUN
21
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
JUN
14
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
JUN
11
Sunday
11:00 AM – 11:50 AM
Cancelled
JUN
11
Sunday
10:00 AM – 10:50 AM
Cancelled
JUN
8
Thursday
12:30 PM – 1:20 PM
Cancelled
JUN
7
Wednesday
12:30 PM – 1:20 PM
Cancelled
JUN
5
Monday
12:30 PM – 1:20 PM
Cancelled
JUN
4
Sunday
10:00 AM – 10:50 AM
Cancelled
JUN
4
Sunday
9:00 AM – 9:50 AM
Cancelled
JUN
2
Friday
12:30 PM – 1:20 PM
Cancelled
MAY
31
Wednesday
12:30 PM – 1:20 PM
Cancelled
MAY
26
Friday
12:30 PM – 1:20 PM
Cancelled
MAY
24
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAY
22
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAY
21
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
MAY
17
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAY
15
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAY
12
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAY
10
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAY
9
Tuesday
11:00 AM – 11:50 AM
Confirmed + $13.94
APR
28
Friday
1:30 PM – 2:20 PM
Confirmed + $13.94
APR
16
Sunday
6:30 AM – 7:20 AM
Confirmed + $13.94
APR
16
Sunday
5:30 AM – 6:20 AM
Confirmed + $13.94
APR
14
Friday
12:00 PM – 12:50 PM
Confirmed + $13.94
APR
14
Friday
11:00 AM – 11:50 AM
Confirmed + $13.94
APR
3
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
APR
1
Saturday
3:30 AM – 4:20 AM
Confirmed + $13.94
MAR
20
Monday
2:00 PM – 2:50 PM
Cancelled
MAR
17
Friday
2:30 PM – 3:20 PM
Confirmed + $13.94
MAR
10
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAR
6
Monday
1:30 PM – 2:20 PM
Confirmed + $13.94
MAR
6
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
FEB
27
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
FEB
26
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
FEB
26
Sunday
9:00 AM – 9:50 AM
Confirmed + $13.94
FEB
20
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
FEB
19
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
FEB
19
Sunday
9:00 AM – 9:50 AM
Confirmed + $13.94
FEB
12
Sunday
6:30 AM – 7:20 AM
Confirmed + $13.94
FEB
12
Sunday
5:30 AM – 6:20 AM
Confirmed + $13.94
FEB
6
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
FEB
5
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
FEB
5
Sunday
9:00 AM – 9:50 AM
Confirmed + $13.94
JAN
30
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
29
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
JAN
27
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
20
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
18
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
17
Tuesday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
15
Sunday
6:00 AM – 6:50 AM
Confirmed + $13.94
JAN
13
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
11
Wednesday
1:30 PM – 2:20 PM
Confirmed + $13.94
JAN
9
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
8
Sunday
10:00 AM – 10:50 AM
Cancelled
JAN
8
Sunday
9:00 AM – 9:50 AM
Cancelled
JAN
6
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
5
Thursday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
4
Wednesday
12:00 PM – 12:50 PM
Confirmed + $13.94
JAN
3
Tuesday
12:00 PM – 12:50 PM
Confirmed + $13.94
JAN
2
Monday
12:00 PM – 12:50 PM
Confirmed + $13.94
DEC
30
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
28
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
26
Monday
12:00 PM – 12:50 PM
Confirmed + $13.94
DEC
26
Monday
11:00 AM – 11:50 AM
Confirmed + $13.94
DEC
23
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
22
Thursday
1:30 PM – 2:20 PM
Confirmed + $13.94
DEC
19
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
18
Sunday
10:00 AM – 10:50 AM
Cancelled
DEC
16
Friday
12:30 PM – 1:20 PM
Cancelled
DEC
14
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
11
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
DEC
11
Sunday
9:00 AM – 9:50 AM
Confirmed + $13.94
DEC
9
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
7
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
5
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
4
Sunday
1:00 PM – 1:50 PM
Confirmed + $13.94
DEC
2
Friday
12:00 PM – 12:50 PM
Confirmed + $13.94
DEC
1
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.94
NOV
29
Tuesday
12:30 PM – 1:20 PM
Cancelled
NOV
27
Sunday
12:00 PM – 12:50 PM
Confirmed + $13.94
NOV
26
Saturday
12:00 PM – 12:50 PM
Confirmed + $13.94
NOV
23
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
NOV
21
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
NOV
20
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
NOV
16
Wednesday
12:30 PM – 1:20 PM
Cancelled
NOV
13
Sunday
1:00 PM – 1:50 PM
Confirmed + $13.94
NOV
11
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
NOV
9
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
NOV
7
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
NOV
6
Sunday
1:00 PM – 1:50 PM
Confirmed + $13.94
NOV
1
Tuesday
2:00 PM – 2:50 PM
Cancelled
OCT
31
Monday
2:00 PM – 2:50 PM
Cancelled
OCT
30
Sunday
3:00 PM – 3:50 PM
Confirmed + $13.94
OCT
30
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
OCT
28
Friday
2:00 PM – 2:50 PM
Confirmed + $13.94
OCT
25
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.94
OCT
23
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
OCT
16
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
OCT
16
Sunday
1:00 PM – 1:50 PM
Confirmed + $13.94
OCT
12
Wednesday
2:00 PM – 2:50 PM
Confirmed + $13.94
OCT
10
Monday
1:00 PM – 1:50 PM
Confirmed + $13.94
OCT
9
Sunday
1:00 PM – 1:50 PM
Confirmed + $13.94
OCT
3
Monday
1:00 PM – 1:50 PM
Confirmed + $13.94
OCT
2
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
30
Friday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
29
Thursday
1:00 PM – 1:50 PM
Confirmed + $13.94
SEP
27
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
26
Monday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
25
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
23
Friday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
23
Friday
1:00 PM – 1:50 PM
Confirmed + $13.94
SEP
14
Wednesday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
8
Thursday
1:00 PM – 1:50 PM
Cancelled
SEP
6
Tuesday
2:00 PM – 2:50 PM
Cancelled
SEP
5
Monday
1:00 PM – 1:50 PM
Cancelled
SEP
4
Sunday
11:00 AM – 11:50 AM
Confirmed + $13.94
AUG
31
Wednesday
12:00 PM – 12:50 PM
Confirmed + $13.94
AUG
28
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
25
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
24
Wednesday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
23
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
19
Friday
1:30 PM – 2:20 PM
Confirmed + $13.94
AUG
18
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
14
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
11
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
9
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
7
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
5
Friday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
28
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
27
Wednesday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
26
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
24
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
24
Sunday
2:00 PM – 2:50 PM
Cancelled
JUL
21
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
19
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
17
Sunday
2:00 PM – 2:50 PM
Cancelled
JUL
15
Friday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUL
12
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUL
10
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUL
10
Sunday
2:00 PM – 2:50 PM
Cancelled
JUL
7
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUL
7
Thursday
2:00 PM – 2:50 PM
Cancelled
JUL
5
Tuesday
2:00 PM – 2:50 PM
Autoconfirmed + $13.26
JUL
3
Sunday
2:00 PM – 2:50 PM
Autoconfirmed + $13.26
JUL
1
Friday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
30
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
28
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
26
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
21
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
19
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
18
Saturday
3:00 PM – 3:50 PM
Confirmed + $13.26
JUN
16
Thursday
2:00 PM – 2:50 PM
Cancelled
JUN
14
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
12
Sunday
3:00 PM – 3:50 PM
Confirmed + $13.26
JUN
12
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
7
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
5
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
3
Friday
2:00 PM – 2:50 PM
Autoconfirmed

]

start_year = 2022  # student start year


SyntaxError: invalid character '–' (U+2013) (1161214193.py, line 5)

In [3]:
import pandas as pd
import datetime
import re

# -----------------------------
# Month mapping
# -----------------------------
month_map = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4, "MAY": 5,
    "JUN": 6, "JUL": 7, "AUG": 8, "SEP": 9, "OCT": 10,
    "NOV": 11, "DEC": 12
}

# -----------------------------
# Parse raw log
# -----------------------------
def parse_raw_log(raw_log, start_year):
    records = []
    for i in range(0, len(raw_log), 5):
        month_str = raw_log[i].strip()
        day = int(raw_log[i+1].strip())
        weekday = raw_log[i+2].strip()
        time_slot = raw_log[i+3].strip()
        status_raw = raw_log[i+4].strip()
        
        # Status
        if "Confirm" in status_raw:
            status = "Confirmed"
        elif "Autoconfirm" in status_raw:
            status = "Confirmed"
        else:
            status = "Cancelled"
        
        # Payment
        payment_match = re.search(r'\$([0-9]+\.[0-9]+)', status_raw)
        payment = float(payment_match.group(1)) if payment_match else 0.0
        
        # Month number
        month = month_map[month_str.upper()]
        
        # Build date
        try:
            date = datetime.date(start_year, month, day)
        except:
            # If invalid date, skip
            continue
        
        records.append({
            "date": date,
            "month": month_str,
            "day": day,
            "weekday": weekday,
            "time_slot": time_slot,
            "status": status,
            "payment": payment
        })
    
    df = pd.DataFrame(records)
    df.sort_values("date", inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

# -----------------------------
# Example usage
# -----------------------------
start_year = 2022

# raw_log = [ "AUG", "30", "Friday", "12:30 PM – 1:20 PM", "Cancelled", ... ] 
# (use your full raw_log here)

df_student = parse_raw_log(raw_log, start_year)
print(df_student.head(10))  # first 10 rows


NameError: name 'raw_log' is not defined

In [4]:
raw_log = [
   AUG
30
Friday
12:30 PM – 1:20 PM
Cancelled
JUN
21
Friday
1:00 PM – 1:50 PM
Cancelled
JUN
10
Monday
1:00 PM – 1:50 PM
Cancelled
MAY
1
Wednesday
12:00 PM – 12:50 PM
Cancelled
APR
25
Thursday
11:00 AM – 11:50 AM
Cancelled
MAR
28
Thursday
12:30 PM – 1:20 PM
Cancelled
FEB
4
Sunday
11:00 AM – 11:50 AM
Confirmed + $13.94
DEC
17
Sunday
11:30 AM – 12:20 PM
Cancelled
DEC
10
Sunday
12:00 PM – 12:50 PM
Cancelled
DEC
10
Sunday
11:00 AM – 11:50 AM
Cancelled
SEP
21
Thursday
12:30 PM – 1:20 PM
Cancelled
SEP
18
Monday
12:30 PM – 1:20 PM
Cancelled
JUL
20
Thursday
12:30 PM – 1:20 PM
Cancelled
JUL
5
Wednesday
11:00 AM – 11:50 AM
Confirmed + $13.94
JUL
3
Monday
11:30 AM – 12:20 PM
Confirmed + $13.94
JUN
30
Friday
12:00 PM – 12:50 PM
Confirmed + $13.94
JUN
25
Sunday
11:30 AM – 12:20 PM
Confirmed + $13.94
JUN
22
Thursday
12:30 PM – 1:20 PM
Confirmed + $13.94
JUN
21
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
JUN
14
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
JUN
11
Sunday
11:00 AM – 11:50 AM
Cancelled
JUN
11
Sunday
10:00 AM – 10:50 AM
Cancelled
JUN
8
Thursday
12:30 PM – 1:20 PM
Cancelled
JUN
7
Wednesday
12:30 PM – 1:20 PM
Cancelled
JUN
5
Monday
12:30 PM – 1:20 PM
Cancelled
JUN
4
Sunday
10:00 AM – 10:50 AM
Cancelled
JUN
4
Sunday
9:00 AM – 9:50 AM
Cancelled
JUN
2
Friday
12:30 PM – 1:20 PM
Cancelled
MAY
31
Wednesday
12:30 PM – 1:20 PM
Cancelled
MAY
26
Friday
12:30 PM – 1:20 PM
Cancelled
MAY
24
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAY
22
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAY
21
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
MAY
17
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAY
15
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAY
12
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAY
10
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAY
9
Tuesday
11:00 AM – 11:50 AM
Confirmed + $13.94
APR
28
Friday
1:30 PM – 2:20 PM
Confirmed + $13.94
APR
16
Sunday
6:30 AM – 7:20 AM
Confirmed + $13.94
APR
16
Sunday
5:30 AM – 6:20 AM
Confirmed + $13.94
APR
14
Friday
12:00 PM – 12:50 PM
Confirmed + $13.94
APR
14
Friday
11:00 AM – 11:50 AM
Confirmed + $13.94
APR
3
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
APR
1
Saturday
3:30 AM – 4:20 AM
Confirmed + $13.94
MAR
20
Monday
2:00 PM – 2:50 PM
Cancelled
MAR
17
Friday
2:30 PM – 3:20 PM
Confirmed + $13.94
MAR
10
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
MAR
6
Monday
1:30 PM – 2:20 PM
Confirmed + $13.94
MAR
6
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
FEB
27
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
FEB
26
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
FEB
26
Sunday
9:00 AM – 9:50 AM
Confirmed + $13.94
FEB
20
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
FEB
19
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
FEB
19
Sunday
9:00 AM – 9:50 AM
Confirmed + $13.94
FEB
12
Sunday
6:30 AM – 7:20 AM
Confirmed + $13.94
FEB
12
Sunday
5:30 AM – 6:20 AM
Confirmed + $13.94
FEB
6
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
FEB
5
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
FEB
5
Sunday
9:00 AM – 9:50 AM
Confirmed + $13.94
JAN
30
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
29
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
JAN
27
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
20
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
18
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
17
Tuesday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
15
Sunday
6:00 AM – 6:50 AM
Confirmed + $13.94
JAN
13
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
11
Wednesday
1:30 PM – 2:20 PM
Confirmed + $13.94
JAN
9
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
8
Sunday
10:00 AM – 10:50 AM
Cancelled
JAN
8
Sunday
9:00 AM – 9:50 AM
Cancelled
JAN
6
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
5
Thursday
12:30 PM – 1:20 PM
Confirmed + $13.94
JAN
4
Wednesday
12:00 PM – 12:50 PM
Confirmed + $13.94
JAN
3
Tuesday
12:00 PM – 12:50 PM
Confirmed + $13.94
JAN
2
Monday
12:00 PM – 12:50 PM
Confirmed + $13.94
DEC
30
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
28
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
26
Monday
12:00 PM – 12:50 PM
Confirmed + $13.94
DEC
26
Monday
11:00 AM – 11:50 AM
Confirmed + $13.94
DEC
23
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
22
Thursday
1:30 PM – 2:20 PM
Confirmed + $13.94
DEC
19
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
18
Sunday
10:00 AM – 10:50 AM
Cancelled
DEC
16
Friday
12:30 PM – 1:20 PM
Cancelled
DEC
14
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
11
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
DEC
11
Sunday
9:00 AM – 9:50 AM
Confirmed + $13.94
DEC
9
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
7
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
5
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
DEC
4
Sunday
1:00 PM – 1:50 PM
Confirmed + $13.94
DEC
2
Friday
12:00 PM – 12:50 PM
Confirmed + $13.94
DEC
1
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.94
NOV
29
Tuesday
12:30 PM – 1:20 PM
Cancelled
NOV
27
Sunday
12:00 PM – 12:50 PM
Confirmed + $13.94
NOV
26
Saturday
12:00 PM – 12:50 PM
Confirmed + $13.94
NOV
23
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
NOV
21
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
NOV
20
Sunday
10:00 AM – 10:50 AM
Confirmed + $13.94
NOV
16
Wednesday
12:30 PM – 1:20 PM
Cancelled
NOV
13
Sunday
1:00 PM – 1:50 PM
Confirmed + $13.94
NOV
11
Friday
12:30 PM – 1:20 PM
Confirmed + $13.94
NOV
9
Wednesday
12:30 PM – 1:20 PM
Confirmed + $13.94
NOV
7
Monday
12:30 PM – 1:20 PM
Confirmed + $13.94
NOV
6
Sunday
1:00 PM – 1:50 PM
Confirmed + $13.94
NOV
1
Tuesday
2:00 PM – 2:50 PM
Cancelled
OCT
31
Monday
2:00 PM – 2:50 PM
Cancelled
OCT
30
Sunday
3:00 PM – 3:50 PM
Confirmed + $13.94
OCT
30
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
OCT
28
Friday
2:00 PM – 2:50 PM
Confirmed + $13.94
OCT
25
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.94
OCT
23
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
OCT
16
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
OCT
16
Sunday
1:00 PM – 1:50 PM
Confirmed + $13.94
OCT
12
Wednesday
2:00 PM – 2:50 PM
Confirmed + $13.94
OCT
10
Monday
1:00 PM – 1:50 PM
Confirmed + $13.94
OCT
9
Sunday
1:00 PM – 1:50 PM
Confirmed + $13.94
OCT
3
Monday
1:00 PM – 1:50 PM
Confirmed + $13.94
OCT
2
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
30
Friday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
29
Thursday
1:00 PM – 1:50 PM
Confirmed + $13.94
SEP
27
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
26
Monday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
25
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
23
Friday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
23
Friday
1:00 PM – 1:50 PM
Confirmed + $13.94
SEP
14
Wednesday
2:00 PM – 2:50 PM
Confirmed + $13.94
SEP
8
Thursday
1:00 PM – 1:50 PM
Cancelled
SEP
6
Tuesday
2:00 PM – 2:50 PM
Cancelled
SEP
5
Monday
1:00 PM – 1:50 PM
Cancelled
SEP
4
Sunday
11:00 AM – 11:50 AM
Confirmed + $13.94
AUG
31
Wednesday
12:00 PM – 12:50 PM
Confirmed + $13.94
AUG
28
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
25
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
24
Wednesday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
23
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
19
Friday
1:30 PM – 2:20 PM
Confirmed + $13.94
AUG
18
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
14
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
11
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
9
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
7
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
AUG
5
Friday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
28
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
27
Wednesday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
26
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
24
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
24
Sunday
2:00 PM – 2:50 PM
Cancelled
JUL
21
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
19
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.94
JUL
17
Sunday
2:00 PM – 2:50 PM
Cancelled
JUL
15
Friday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUL
12
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUL
10
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUL
10
Sunday
2:00 PM – 2:50 PM
Cancelled
JUL
7
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUL
7
Thursday
2:00 PM – 2:50 PM
Cancelled
JUL
5
Tuesday
2:00 PM – 2:50 PM
Autoconfirmed + $13.26
JUL
3
Sunday
2:00 PM – 2:50 PM
Autoconfirmed + $13.26
JUL
1
Friday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
30
Thursday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
28
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
26
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
21
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
19
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
18
Saturday
3:00 PM – 3:50 PM
Confirmed + $13.26
JUN
16
Thursday
2:00 PM – 2:50 PM
Cancelled
JUN
14
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
12
Sunday
3:00 PM – 3:50 PM
Confirmed + $13.26
JUN
12
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
7
Tuesday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
5
Sunday
2:00 PM – 2:50 PM
Confirmed + $13.26
JUN
3
Friday
2:00 PM – 2:50 PM
Autoconfirmed

]

start_year = 2022  # student start year


SyntaxError: invalid character '–' (U+2013) (1161214193.py, line 5)